# Imports

In [1]:
import pandas as pd
import numpy as np
import os
import requests
import time
import math
from tqdm import tqdm

In [2]:
idx = -1

my_states = ["New Mexico", "Massachusetts", "Ohio", "Texas", "Florida", "Washington", "Illinois", "New Mexico"]
state_name = my_states[idx]


my_cities = ["Albuquerque", "Boston", "Cleveland", "Dallas", "Miami", "Seattle", "Chicago", "Santa Fe"]
city_name = my_cities[idx]

api_keys = [
    # 'd927fb04-31ab-43e6-9b4b-62a11d96e928', # GA
    # '1b08644e-8836-4d6b-86b6-22ed8b02a4c6', # GA
    'aabc2a52-f125-48a7-a9af-f2cb7530fcaf', # kulacs@miamioh.edu
    '9bbfedba-03f7-4885-a883-3a633fa55340', # colin.kulaa@gmail.com
    '30fc1af0-df20-4544-9da9-b96d28e64550', # mkw1612@gmail.com
    '1e49a427-6a1e-48bd-abf8-983ed9cc17aa', # walke262@miamioh.edu
    '4ab28b56-58c4-47c9-9c07-38d82bb09ad0', # klingegm@miamioh.edu
    '8b8be3ca-9ceb-4efe-a128-4ede4934de52', # riceaa4@miamioh.edu
    '395b6e4e-6584-4465-8be7-66c475f50f12', # aubreyrice04@gmail.com
]

GLOBAL_API_FAILURE = False

key_idx = 0

request_limit = 1

route_url = "https://graphhopper.com/api/1/route"

modes = ["car"]# , "bike", "foot"]

CHECKPOINT_FREQ = 5 # Save every 5 edges

# Setup/Loading Data

In [3]:
# Ensure working directory is repo root (parent of city_data_generation/)
if os.path.basename(os.getcwd()) == "city_data_generation":
    os.chdir("..")

data_dir = os.path.join("data_real", city_name.lower().replace(" ", "_"))
edges_file = data_dir + "/template_optimal_edges_list.csv"

# Load data - ensure dtypes match your CSV format
edges_df = pd.read_csv(edges_file, dtype={'from_node': str, 'to_node': str, 'distance_km': float, 'time_hours': float, 'weight': float, 'lat_from': float, 'lon_from': float, 'lat_to': float, 'lon_to': float})
print(f"Original edges_df columns: {list(edges_df.columns)}")

Original edges_df columns: ['from_node', 'to_node', 'distance_km', 'time_hours', 'weight', 'lat_from', 'lon_from', 'lat_to', 'lon_to']


# Query Function

In [4]:
def gh_route_query(lon_a, lat_a, lon_b, lat_b, profile):
    global key_idx

    while True:
        params = {
            "key": api_keys[key_idx],
            "profile": profile,
            # The locations will alternate every call
            # Forward is to-> from
            # Reverse is from -> to
            "point": [f"{lat_a},{lon_a}", f"{lat_b},{lon_b}"],
            "calc_points": "false" # Keep payload small
        }

        # Get the response based on params
        response = requests.get(route_url, params=params)
        data = response.json()

        # Success
        if "message" not in data:
            return data

        # Error handling
        msg = data["message"]
        print(f"Route Error: {msg}")

        # Rate limiting
        if "minutely api limit" in msg.lower() or "too many requests" in msg.lower():
            print("\tHit rate limit — waiting 60 seconds...")
            time.sleep(60)

        # API key daily limit check
        elif "limit" in msg.lower():
            if key_idx < len(api_keys) - 1:
                key_idx += 1
                print(f"\nSWITCHING TO NEXT KEY: {key_idx}\n")
            else:
                # THIS IS THE GLOBAL FAILURE POINT
                GLOBAL_API_FAILURE = True
                print(f"\tFATAL ERROR: All API keys exhausted or locked out. Stopping global execution.")
                return None

        # Other errors
        else:
            # Log it and break
            print(f"\tUnrecoverable error: {msg}. Breaking loop for current edge.")
            return None

# Edge Processing

In [5]:
for mode in modes:
    print(f"\nProcessing mode: {mode.upper()}\n")
    output_file = f"{data_dir}/edges_{mode}_bidirectional.csv"

    # Check if we are resuming or starting from scratch
    try:
        # Load the partially processed file
        updated_edges = pd.read_csv(output_file, dtype={'from_node': str, 'to_node': str})
        print(f"Resuming work from existing file: {output_file}")
    except FileNotFoundError:
        # Start from the template and initialize columns
        updated_edges = edges_df.copy()
        # Initialize forward columns to nan (will be overwritten)
        updated_edges["distance_km"] = np.nan
        updated_edges["time_hours"] = np.nan
        updated_edges["weight"] = np.nan
        # Initialize reverse columns to nan (will be overwritten)
        updated_edges["distance_km_reverse"] = np.nan
        updated_edges["time_hours_reverse"] = np.nan
        updated_edges["weight_reverse"] = np.nan
        print(f"Starting new processing file at: {output_file}")

    n = len(updated_edges)

    # Find the starting index. This is the first row where 'distance_km_reverse' is nan
    # We use distance_km_reverse as the reliable flag for a completely processed row.
    start_index_series = updated_edges['distance_km_reverse'].isnull()

    try:
        # Keep rows where the series value is True
        # The first index of the list is the starting index we will use.
        start_index = start_index_series[start_index_series].index[0]
        print(f"Starting from index: {start_index} out of {n}")
    except IndexError:
        start_index = n
        print(f"All {n} edges already processed for mode '{mode}'.")

    if start_index >= n:
        print("Data is already processed. Move to next mode.")
        continue

    # Main processing loop using the starting index we just found
    for idx in tqdm(range(start_index, n), desc=f"{mode} edges", initial=start_index, total=n):

        # Check for global halt signal
        if GLOBAL_API_FAILURE:
            print("\nExiting loop due to global API failure.")
            break

        # Check if the row is already complete before making any API calls
        if not pd.isna(updated_edges.loc[idx, "distance_km"]) and not pd.isna(updated_edges.loc[idx, "distance_km_reverse"]):
            # This row is already complete (ran successfully in a previous session)
             continue # Skip the rest of the loop for this index

        # Extract the coordinate pairs for the current edge
        lon_from, lat_from = updated_edges.loc[idx, ["lon_from", "lat_from"]]
        lon_to, lat_to = updated_edges.loc[idx, ["lon_to", "lat_to"]]

        # FORWARD route (from -> to)
        result_fwd = gh_route_query(lon_from, lat_from, lon_to, lat_to, mode)

        if result_fwd and "paths" in result_fwd and result_fwd["paths"]:
            path = result_fwd["paths"][0]
            dist_m_fwd = path.get("distance", 0)    # meters
            time_ms_fwd = path.get("time", 0)       # milliseconds

            # The /route endpoint DOES NOT return the weight
            # Our replacement for weight is time in seconds
            time_s_fwd = time_ms_fwd / 1000

            # Save forward results
            updated_edges.loc[idx, "distance_km"] = dist_m_fwd / 1000
            updated_edges.loc[idx, "time_hours"] = time_s_fwd / 3600
            updated_edges.loc[idx, "weight"] = time_s_fwd

        # Check for global halt signal
        if GLOBAL_API_FAILURE:
            print("\nExiting loop due to global API failure.")
            break

        # REVERSE route (to -> from)
        result_rev = gh_route_query(lon_to, lat_to, lon_from, lat_from, mode)

        if result_rev and "paths" in result_rev and result_rev["paths"]:
            path = result_rev["paths"][0]
            dist_m_rev = path.get("distance", 0)
            time_ms_rev = path.get("time", 0)

            time_s_rev = time_ms_rev / 1000

            # Save reverse results
            updated_edges.loc[idx, "distance_km_reverse"] = dist_m_rev / 1000
            updated_edges.loc[idx, "time_hours_reverse"] = time_s_rev / 3600
            updated_edges.loc[idx, "weight_reverse"] = time_s_rev

        else:
            # If reverse fails, we might still have a partial forward result, but
            # we should log the failure and let the loop continue.
            # These columns remain nan so this can be identified
            print(f"Index {idx}: Reverse route to->from failed to retrieve data.")


        # Checkpoint save logic
        # Saves every 5, ex. idx 0 to idx 4
        if (idx + 1) % CHECKPOINT_FREQ == 0:
            updated_edges.to_csv(output_file, index=False)
            print(f"\nCheckpoint saved at index {idx+1}")

        # Slow down slightly to stay under rate limits.
        time.sleep(10)

    # Final save output
    updated_edges.to_csv(output_file, index=False)
    print(f"\nFinal results saved to {output_file}")

    # Check if the modes loop should stop
    if GLOBAL_API_FAILURE:
        print("\n\n!! GLOBAL HALT !! Stopping all further processing due to API key exhaustion.")
        break


Processing mode: CAR

Starting new processing file at: data_real/santa_fe/edges_car_bidirectional.csv
Starting from index: 0 out of 155


car edges:   3%|▎         | 4/155 [00:43<27:18, 10.85s/it]

Route Error: Minutely API limit heavily violated. Skip request. Find the details at the bottom of the page https://www.graphhopper.com/pricing/
	Hit rate limit — waiting 60 seconds...

Checkpoint saved at index 5


car edges:   6%|▌         | 9/155 [02:38<37:40, 15.48s/it]  

Route Error: Minutely API limit heavily violated. Skip request. Find the details at the bottom of the page https://www.graphhopper.com/pricing/
	Hit rate limit — waiting 60 seconds...

Checkpoint saved at index 10


...

Route Error: Minutely API limit heavily violated. Skip request. Find the details at the bottom of the page https://www.graphhopper.com/pricing/
	Hit rate limit — waiting 60 seconds...


car edges:  99%|█████████▉| 154/155 [56:14<00:18, 18.62s/it]


Checkpoint saved at index 155


car edges: 100%|██████████| 155/155 [56:24<00:00, 21.84s/it]


Final results saved to data_real/santa_fe/edges_car_bidirectional.csv


# Tests

In [6]:
# Load the final processed file
output_file = f"{data_dir}/edges_car_bidirectional.csv"
processed_df = pd.read_csv(output_file)# , nrows=700)

print("\nData Quality Checks")

# Check 1: Forward distance > 0 and not nan
valid_fwd = processed_df['distance_km'].dropna()
print(f"Valid forward edges: {len(valid_fwd)} / {len(processed_df)}")
assert len(valid_fwd) > 0.99 * len(processed_df), "Too many missing forward distances."

# Check 2: Reverse distance > 0 and not nan
valid_rev = processed_df['distance_km_reverse'].dropna()
print(f"Valid reverse edges: {len(valid_rev)} / {len(processed_df)}")
assert len(valid_rev) > 0.99 * len(processed_df), "Too many missing reverse distances."

# Check 3: Check for asymmetry
symmetric_distances = np.isclose(processed_df['distance_km'], processed_df['distance_km_reverse'], equal_nan=True)
symmetric_times = np.isclose(processed_df['time_hours'], processed_df['time_hours_reverse'], equal_nan=True)

num_symmetric_dist = symmetric_distances.sum()
num_symmetric_time = symmetric_times.sum()

print(f"Edges with symmetric distance: {num_symmetric_dist} ({(num_symmetric_dist/len(processed_df))*100:.2f}%)")
print(f"Edges with symmetric Ttme: {num_symmetric_time} ({(num_symmetric_time/len(processed_df))*100:.2f}%)")

# A healthy urban road network should have significant asymmetry
assert num_symmetric_time < 0.9 * len(processed_df), "Too many symmetric times, check data or graphhopper profile."

# Check 4: Reasonable speed check (ex. 5 km/h minimum)
# Calculate average forward speed (km/h)
processed_df['avg_speed_kph'] = processed_df['distance_km'] / processed_df['time_hours']
slowest_speed = processed_df['avg_speed_kph'].min()
fastest_speed = processed_df['avg_speed_kph'].max()

print(f"Min avg speed: {slowest_speed:.2f} kph")
print(f"Max avg speed: {fastest_speed:.2f} kph")

assert slowest_speed > 5, "Implausibly slow speed detected (likely walking speed or bad data)."


Data Quality Checks
Valid forward edges: 155 / 155
Valid reverse edges: 155 / 155
Edges with symmetric distance: 2 (1.29%)
Edges with symmetric Ttme: 2 (1.29%)
Min avg speed: 18.68 kph
Max avg speed: 57.37 kph
